# LegaLoom-Env: Full Curriculum GRPO Training

Trains across **two phases** (easy → hard), 20 GRPO steps per phase = **40 total steps**. Uses Unsloth + TRL on Colab T4.

**Runtime:** ~90-120 min on Colab T4 GPU.

**Outputs:** `reward_curves.png`, `before_after.png`, `training_log.json`, `training_scores.json`

In [ ]:
# Cell 1: Install
!pip install unsloth 'trl>=0.12.0' openenv-core==0.2.3 fastapi uvicorn pydantic httpx openai pyyaml datasets matplotlib --quiet
print('✓ Installed')

In [ ]:
# Cell 2: Clone repo
import sys, os, subprocess

REPO_DIR = '/content/legaloom_env'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone',
                    'https://huggingface.co/spaces/aarav0202/legaloom-env',
                    REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'✓ Working dir: {os.getcwd()}')

In [ ]:
# Cell 3: Load model + LoRA via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print('✓ Model loaded')

In [ ]:
# Cell 4: Baseline measurement — 10 episodes per task
from unsloth import FastLanguageModel as FLM
from train_grpo import rollout_episode
from server.legaloom_env_environment import LegaloomEnvironment

FLM.for_inference(model)

print('Measuring baseline (untrained), 10 episodes per task...')
baseline_scores = {}
baseline_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    baseline_scores[task_id] = sum(scores) / len(scores)
    baseline_per_episode[task_id] = scores
    std = (sum((s - baseline_scores[task_id])**2 for s in scores) / len(scores))**0.5
    print(f'  {task_id}: mean={baseline_scores[task_id]:.3f}  std={std:.3f}')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')

In [ ]:
# Cell 5: 2-phase GRPO training (40 steps total)
from unsloth import FastLanguageModel as FLM
from train_grpo import run_curriculum_training

FLM.for_training(model)

schedule = ['task_easy', 'task_hard']  # 2-phase: matches README results
log_history = run_curriculum_training(
    model=model,
    tokenizer=tokenizer,
    task_schedule=schedule,
    steps_per_phase=20,
    examples_per_task=60,
    max_prompt_length=1536,
    max_completion_length=768,
)
print(f'\n✓ Done. {len(log_history)} log entries across {len(schedule)} phases.')

In [ ]:
# Cell 6: Plot reward curves with phase boundaries
import matplotlib.pyplot as plt
import json

with open('training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2, default=str)

rewards, phases, phase_tasks = [], [], []
for e in log_history:
    if 'reward' in e:
        rewards.append(e['reward'])
        phases.append(e.get('phase', 0))
        phase_tasks.append(e.get('phase_task_id', '?'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = list(range(1, len(rewards) + 1))
ax.plot(x, rewards, 'b-', linewidth=1.5, alpha=0.6, label='Episode reward')

# Phase boundaries
phase_colors = ['#e8f4ff', '#e8ffe8', '#fff5e8', '#ffe8e8']
phase_starts = [0]
cur = phases[0] if phases else 0
for i, p in enumerate(phases):
    if p != cur:
        phase_starts.append(i)
        cur = p
phase_starts.append(len(rewards))
for i in range(len(phase_starts) - 1):
    pi = phases[phase_starts[i]] if phase_starts[i] < len(phases) else 0
    ax.axvspan(phase_starts[i] + 1, phase_starts[i + 1], color=phase_colors[pi % 4], alpha=0.4)
    mid = (phase_starts[i] + phase_starts[i + 1]) / 2 + 1
    lbl = phase_tasks[phase_starts[i]].replace('task_', '') if phase_starts[i] < len(phase_tasks) else '?'
    ax.text(mid, 0.95, lbl, ha='center', va='top', fontsize=10, fontweight='bold')

if len(rewards) >= 3:
    w = 3
    ma = [sum(rewards[max(0, j-w+1):j+1]) / min(j+1, w) for j in range(len(rewards))]
    ax.plot(x, ma, 'r-', linewidth=2.5, label='3-step moving avg')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xlabel('Training Step'); ax.set_ylabel('Episode Reward')
ax.set_title('GRPO Curriculum — Reward across 4 phases')
ax.set_ylim(0, 1); ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)

ax2 = axes[1]
losses = [e['loss'] for e in log_history if 'loss' in e]
if losses:
    ax2.plot(range(1, len(losses)+1), losses, 'g-', linewidth=1.5, alpha=0.8)
ax2.set_xlabel('Training Step'); ax2.set_ylabel('Loss')
ax2.set_title('GRPO Curriculum — Loss'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ reward_curves.png saved')

In [ ]:
# Cell 7: Trained model evaluation — 10 episodes per task
FLM.for_inference(model)

print('Measuring trained model, 10 episodes per task...')
trained_scores = {}
trained_per_episode = {}
for task_id in ['task_easy', 'task_medium', 'task_hard', 'task_expert']:
    scores = []
    for seed in range(42, 52):
        env = LegaloomEnvironment()
        result = rollout_episode(model, tokenizer, env, task_id, seed=seed, temperature=0.1)
        scores.append(result['final_reward'])
    trained_scores[task_id] = sum(scores) / len(scores)
    trained_per_episode[task_id] = scores
    std = (sum((s - trained_scores[task_id])**2 for s in scores) / len(scores))**0.5
    delta = trained_scores[task_id] - baseline_scores[task_id]
    rel = (delta / baseline_scores[task_id] * 100) if baseline_scores[task_id] > 0 else 0
    print(f'  {task_id}: before={baseline_scores[task_id]:.3f}  after={trained_scores[task_id]:.3f}  std={std:.3f}  Δ={delta:+.3f} ({rel:+.0f}%)')

print(f'\nBaseline avg: {sum(baseline_scores.values()) / 4:.3f}')
print(f'Trained avg:  {sum(trained_scores.values()) / 4:.3f}')
print(f'Lift:         {(sum(trained_scores.values()) - sum(baseline_scores.values())) / 4:+.3f}')

In [ ]:
# Cell 8: Before/After bar chart + save scores
import matplotlib.pyplot as plt
import json

tasks = ['task_easy', 'task_medium', 'task_hard', 'task_expert']
labels = ['Easy', 'Medium', 'Hard', 'Expert']
before = [baseline_scores[t] for t in tasks]
after = [trained_scores[t] for t in tasks]

fig, ax = plt.subplots(figsize=(11, 6))
x = range(len(tasks)); w = 0.36
ax.bar([i - w/2 for i in x], before, w, label='Before GRPO (untrained)', color='#e76f51')
ax.bar([i + w/2 for i in x], after, w, label='After GRPO (2-phase GRPO)', color='#2a9d8f')
for i in x:
    ax.text(i - w/2, before[i] + 0.015, f'{before[i]:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, after[i] + 0.015, f'{after[i]:.3f}', ha='center', fontsize=9)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Success threshold')
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel('Average Score (10 episodes)')
ax.set_title('LegaLoom-Env: Before vs After Curriculum GRPO Training')
ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
plt.show()

with open('training_scores.json', 'w') as f:
    json.dump({
        'baseline': baseline_scores,
        'trained': trained_scores,
        'baseline_per_episode': baseline_per_episode,
        'trained_per_episode': trained_per_episode,
        'n_episodes_per_task': 10,
        'training_schedule': schedule,
        'steps_per_phase': 20,
    }, f, indent=2)
print('✓ before_after.png + training_scores.json saved')

In [ ]:
# Cell 9: Print README table + download artifacts
from google.colab import files

print('=' * 60)
print('PASTE THIS TABLE INTO README.md Results section:')
print('=' * 60)
print()
print('| Task | Baseline | After GRPO | Δ |')
print('|------|---------:|-----------:|------:|')
for tid, lbl in [('task_easy','`task_easy`'),('task_medium','`task_medium`'),
                  ('task_hard','`task_hard`'),('task_expert','`task_expert`')]:
    b, a = baseline_scores[tid], trained_scores[tid]
    d = ((a-b)/b*100) if b > 0 else 0
    bold = '**' if a > b else ''
    print(f'| {lbl} | {b:.3f} | {bold}{a:.3f}{bold} | {d:+.0f}% |')
ab = sum(baseline_scores.values())/4
aa = sum(trained_scores.values())/4
ad = ((aa-ab)/ab*100) if ab > 0 else 0
print(f'| **Average** | **{ab:.3f}** | **{aa:.3f}** | **{ad:+.0f}%** |')
print()
print('Also update the Setup paragraph: "40 GRPO steps — 2-phase (easy + hard)"')
print('=' * 60)
print()
print('Downloading artifacts...')
for f_name in ['reward_curves.png', 'before_after.png', 'training_scores.json', 'training_log.json']:
    try:
        files.download(f_name)
        print(f'  ✓ {f_name}')
    except Exception as e:
        print(f'  ✗ {f_name}: {e}')
print('\nDrop these 4 files into your repo root, update README table, commit, push.')